# Observations and output sites

Define the onshore wave observations available for your case (buoys, tide gauges, spectral sites, ...), preview them on a map, and save a **JSON (GeoJSON format)** file compatible with QGIS and the HyGloW config loader.

Workflow:

1. Set the output file name/path in the parameters cell.
2. Edit the `observations` dict below (one entry per instrument).
3. Run the preview cells to check table and map.
4. Save the JSON to disk.
5. Add/update `"onshore_waves"` in the case JSON with the saved file path.
6. Build the output sites on the model mesh (isobaths + buoys) and check
   their mesh depths: the metamodel and dynamic runs use this table.

In [ ]:
from utils.config import PATHS, resolve
from utils import onshore_sites as obs
from utils import viz as plot

## Parameters

In [ ]:
from utils.config import ONSHORE_WAVES

# Output file — defaults to the path declared in the case config ("onshore_waves").
output_observations_file = ONSHORE_WAVES
output_path = resolve(output_observations_file)

print(f"output   : {output_path}")
print(f"AoI      : {resolve(PATHS['aoi_file'])}")
print(f"topobathy: {resolve(PATHS['topobathy_file'])}")

## Observations dict

Edit the dictionary below. Keys are unique `instrument_id` strings.

Required per entry:
- `lon`, `lat` — WGS84 coordinates
- `type` — role used downstream (`validation`, `waves`, `spectrum`, `calibration`, `levels`, `ignore`, …)
- `institution` — data provider (e.g. `IMOS`, `NDBC`, `SPREP`)
- `recording` — approximate coverage period as a string (e.g. `"2010-2024"`)

Optional:
- `measured_depth` — water depth from the station metadata, m positive down
  (`null` if unknown); the model uses the mesh depth (see the end of this notebook)
- `name` — human-readable station name
- `role` — overrides `type` when filtering by role in later notebooks
- any extra property (e.g. `msl` for tide gauges) is kept in the GeoJSON

In [ ]:
observations: dict[str, dict] = {
    "Aunuu_1": {
        "lon": -170.4933,
        "lat": -14.265,
        "type": "waves",
        "institution": "CDIP / PacIOOS (station 189, NDBC 51209)",
        "recording": "2014-10 to 2018-04",
        "measured_depth": 79.0,
        "name": "Aunu'u, American Samoa (CDIP 189, first site)",
    },
    "Aunuu_2": {
        "lon": -170.5005,
        "lat": -14.2732,
        "type": "waves",
        "institution": "CDIP / PacIOOS (station 189, NDBC 51209)",
        "recording": "2018-05 to present",
        "measured_depth": 55.0,
        "name": "Aunu'u, American Samoa (CDIP 189, second site)",
    },
    "King_Poloa": {
        "lon": -170.8747,
        "lat": -14.2965,
        "type": "waves",
        "institution": "CDIP / PacIOOS (station 273, NDBC 51214)",
        "recording": "2024-06 to present",
        "measured_depth": 60.0,
        "name": "King Poloa, Tutuila, American Samoa (CDIP 273)",
    },
}

## Build GeoJSON and preview table

In [ ]:
feature_collection = obs.dict_to_geojson(observations)
preview_df = obs.geojson_to_dataframe(feature_collection)
preview_df

## Map

In [ ]:
plot.plot_case_map(
    obs.dict_to_gdf(observations), title="Case observations", show_satellite=True
)

## Save observations JSON (GeoJSON format)

In [ ]:
obs.save_geojson(feature_collection, output_path)

print(f"Wrote {len(feature_collection['features'])} feature(s) to:\n  {output_path}")

## Processed buoy records (CDIP)

The buoys with data are the CDIP / PacIOOS stations of American Samoa
(searched 2026-11: no other wave buoy with data in the domain; the 1989-93
Western Samoa Waveriders survive only as reports). Each CDIP station file
(archive + real-time) becomes hourly `hs`, `tp`, `tm01`, `dir`, `spr` and the 2D
spectrum `efth` (`bluemath_tk.waves.cdip.cdip_hourly` / `cdip_hourly_spectra`: spectrum and
a1, b1, a2, b2 moments averaged per hour, good records only; `efth` = E(f)
spread by the Maximum Entropy Method, as CDIP does) and every hour goes to the site its GPS
fix is within `max_distance_km` of: Aunu'u was moored at two sites (first site
to 2018-04, second from 2018-05).

Written to `inputs/Observations/processed/<instrument_id>.nc` (bulk, read by
10) and `processed/spectra/<instrument_id>.nc` (`efth`, read by 11;
`val.buoy_spectra_path`).

In [ ]:
import numpy as np
import xarray as xr
from bluemath_tk.waves.cdip import cdip_hourly, cdip_hourly_spectra, cdip_url
from utils.validation import buoy_spectra_path

# CDIP files read: station -> archive (False) and/or real-time (True) file
cdip_files = {189: [False, True], 273: [True]}
max_distance_km = 0.6  # an hour belongs to the site its GPS fix is within
dirs = np.arange(0.0, 360.0, 5.0)  # directions of the 2D spectra (deg, coming from)

hourly, spectra = [], []
for st, flags in cdip_files.items():
    for rt in flags:
        raw = xr.open_dataset(cdip_url(st, realtime=rt)).load()
        hourly.append(cdip_hourly(raw))
        spectra.append(cdip_hourly_spectra(raw, dirs))
hourly = xr.concat(hourly, dim="time")
spectra = xr.concat(spectra, dim="time")  # stations differ in frequency bands: NaN where a band is missing

processed_dir = resolve("inputs/Observations/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
for iid, site in observations.items():
    km = np.hypot(
        (hourly["longitude"] - site["lon"]) * 111.32 * np.cos(np.radians(site["lat"])),
        (hourly["latitude"] - site["lat"]) * 110.57,
    )
    keep = (km < max_distance_km).values
    record = hourly.isel(time=keep).sortby("time")
    record = record.isel(time=~record.indexes["time"].duplicated())
    spec = spectra.isel(time=keep).sortby("time")
    spec = spec.isel(time=~spec.indexes["time"].duplicated()).dropna("freq", how="all")
    attrs = hourly.attrs | dict(station_id=iid, lon=site["lon"], lat=site["lat"], max_distance_km=max_distance_km)
    record[["hs", "tp", "tm01", "dir", "spr"]].assign_attrs(attrs).to_netcdf(processed_dir / f"{iid}.nc")
    spectra_path = buoy_spectra_path(iid)
    spectra_path.parent.mkdir(parents=True, exist_ok=True)
    spec.assign_attrs(attrs).to_netcdf(spectra_path, encoding={"efth": {"zlib": True, "complevel": 4}})
    print(f"{iid:11s} {record.sizes['time']:6d} h  {str(record.time.values[0])[:10]} .. {str(record.time.values[-1])[:10]}"
          f"  spectra {spec.sizes['freq']} bands")

## Update the case JSON

After saving, add or update this key in the case JSON so downstream notebooks use your new file:

```json
"onshore_waves": "inputs/Observations/<Case>_observations.json"
```

Use the exact relative path you set in `output_observations_file`.

## Output sites on the model mesh

All output sites are built here, once, on the mesh of `simulations.model`
(`simulations.<model>.grid_file`): isobaths traced on the topobathy raster
blanked outside the mesh (`simulations.output_sites_config`), plus the buoys
just saved. Each site gets its **`mesh_depth`** (the depth the models see, used
from here on) next to its nominal depth (contour level, or the buoy's
`measured_depth`), and is kept only if it is inside the mesh and still wet at
the lowest metamodel water level (`simulations.bounds.wl[0]`, with
`simulations.min_depth`).

The table, dropped sites included, goes to `outputs/output_sites_{model}.csv`;
the metamodel cases (05) and the dynamic runs (08) read it. Rebuilding it
(`recompute=True`) invalidates runs made with the old sites.

In [ ]:
from utils.geometry import buoy_depth_check, output_sites, sites_summary

sites = output_sites(recompute=True, all_sites=True)
sites_summary(sites)

Buoys: measured depth (station metadata) against the mesh depth. A large
difference (`flag`) points to wrong coordinates, a bank or channel the mesh
smooths out, or another vertical datum.

In [ ]:
buoy_depth_check(sites)

In [ ]:
plot.plot_output_sites(sites, show_satellite=True)
plot.plot_site_depths(sites);